In [7]:
from src.qwen import load_qwen

model, tokenizer = load_qwen()

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


# Look at Model Configuration

In [8]:
print(model)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2RotaryEmbe

In [9]:
model.config

Qwen2Config {
  "_attn_implementation_autoset": true,
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 896,
  "initializer_range": 0.02,
  "intermediate_size": 4864,
  "max_position_embeddings": 32768,
  "max_window_layers": 21,
  "model_type": "qwen2",
  "num_attention_heads": 14,
  "num_hidden_layers": 24,
  "num_key_value_heads": 2,
  "rms_norm_eps": 1e-06,
  "rope_scaling": null,
  "rope_theta": 1000000.0,
  "sliding_window": 32768,
  "tie_word_embeddings": true,
  "torch_dtype": "float32",
  "transformers_version": "4.50.0",
  "use_cache": true,
  "use_sliding_window": false,
  "vocab_size": 151936
}

# Get General Idea on the FLOPS in this project

In [2]:
from src.FLOPS import total_flops
import numpy as np

# Calculate the FLOPS without LoRA
total_flops_for_whole_sample = total_flops(1, 1, (896, 512), 14, 4864, 151936)

# Single Forward Pass with 1 step and batch size of 1
print(f"Total FLOPS for Single Forward Pass (1 step, batch size 1): {total_flops_for_whole_sample / (3 * 1e12):.3f} TFLOPS")

# Forward + Backward Pass with 1 step and batch size of 1
print(f"Total FLOPS for Forward + Backward Pass (1 step, batch size 1): {total_flops_for_whole_sample / 1e12:.3f} TFLOPS")

/Users/yilan/Desktop/Cambridge/venvs/m2env/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Total FLOPS for Single Forward Pass (1 step, batch size 1): 0.509 TFLOPS
Total FLOPS for Forward + Backward Pass (1 step, batch size 1): 1.527 TFLOPS


In [ ]:
from src.FLOPS import lora_flops_forward, total_flops_lora

# Calculate the FLOPS with LoRA

# LoRA Addition FLOPS, Single Forward Pass with 1 step and batch size of 1
lora_addition_flops = lora_flops_forward((896, 512), 14, 4) / 1e8
print(f"LoRA Addition FLOPS (Single Forward Pass, 1 step, batch size 1): {lora_addition_flops:.3f} e8 FLOPS")

# Total FLOPS of the LoRA-tuned model, Forward + Backward Pass with 1 step and batch size of 1
total_lora_flops = total_flops_lora(1, 1, (896, 512), 14, 4864, 151936, 4) / 1e12
print(f"Total FLOPS for LoRA-tuned model (Forward + Backward Pass, 1 step, batch size 1): {total_lora_flops:.3f} TFLOPS")

LoRA Addition FLOPS (Single Forward Pass, 1 step, batch size 1): 3.950 GFLOPS
Total FLOPS for LoRA-tuned model (Forward + Backward Pass, 1 step, batch size 1): 1.528 TFLOPS


# Calculate the FLOPS count for this Project

In [13]:
# Before Training: Overfit Test
flops_overfit = total_flops_lora(1000, 4, (896,512), 14, 4864, 151936, 4)
print(flops_overfit/(1*(1e17)))

0.06113081100288


In [14]:
# 3(a) FLOPS: LORA Default
LORA_defalt = total_flops_lora(3000, 4, (896,512), 14, 4864, 151936, 4)
print(LORA_defalt/(1*(1e17)))

0.18339243300864


In [15]:
# 3(b) FLOPS: learning rate & LoRA rank 3x3 grid search
# Learning rate in (10−5, 5 × 10−5, 1 × 10−4)
# LoRA rank in (2, 4, 8)

# Only rank affects the FLOPS count
flops_rank2 = total_flops_lora(600, 4, (896,512), 14, 4864, 151936, 2)
flops_rank4 = total_flops_lora(600, 4, (896,512), 14, 4864, 151936, 4)
flops_rank8 = total_flops_lora(600, 4, (896,512), 14, 4864, 151936, 8)

flops_3b_1 = (flops_rank2+flops_rank4+flops_rank8)*3

print(flops_3b_1/(1*(1e17)))

0.330144281505792


In [16]:
# 3(b) FLOPS: context length tuning x 3
# context length in (128, 512, 768)

flops_ctx_128 = total_flops_lora(600, 4, (896,128), 14, 4864, 151936, 8)
#flops_ctx_512 = total_flops_lora(600, 4, (896,512), 14, 4864, 151936, 8)
flops_ctx_768 = total_flops_lora(600, 4, (896,768), 14, 4864, 151936, 8)

flops_3b_2 = flops_ctx_128+flops_ctx_768

print(flops_3b_2/(1*(1e17)))

0.065184027890688


In [17]:
# 3(c) FLOPS: Final Training with optimal hyperparameters
flops_final_train = total_flops_lora(3000, 4, (896,768), 14, 4864, 151936, 8)

print(flops_final_train/(1*(1e17)))

0.28162787484672


In [18]:
# Total FLOPS
total_flops = ( flops_overfit + LORA_defalt + flops_3b_1 + flops_3b_2 + flops_final_train)/1e17
print("Total FLOPS: ", total_flops)

Total FLOPS:  0.92147942825472
